# Fase A — Modelo base con datos reales (CNN de caracteres)
**Sistema Predictivo Basado en Deep Learning para la Detección de Phishing en Usuarios Digitales de Arequipa**
Keny Russell Guerra Huanaco — UCSM, Ingeniería de Sistemas — Asesor: Nestor Torres

Ejecuta **una celda a la vez, de arriba hacia abajo**, y espera a que termine (aparece un número entre corchetes a la izquierda). No saltes pasos.

| Paso | Qué hace | Evidencia que genera |
|---|---|---|
| 0 | Verifica GPU y versiones | versiones del entorno |
| 1 | Instala dependencias y sube el proyecto | — |
| 2 | Descarga PhiUSIIL y balancea 80,000 URLs (RNF6) | `data/raw/dataset_unificado.csv` |
| 3 | Análisis exploratorio (calidad de datos y posibles atajos) | `evidencias/eda_*` |
| 4 | Preprocesamiento (tokenización por carácter, 70/10/20) | `data/processed/` |
| 5 | Entrenamiento con validación cruzada k=10 | `models/cnn_phishing_kfold_metrics.json` |
| 6 | Evaluación en test nunca visto + prueba de robustez | `evidencias/` |
| 7 | Exportación a ONNX + latencia (RNF1) | `models/cnn_phishing.onnx` |
| 8 | Resumen y descarga de todo | `fase_A_resultados.zip` |

**Importante:** todas las cifras que irán a tu informe son las que imprima *este* notebook al ejecutarlo; no uses números de pruebas anteriores (eran datos sintéticos).

## Paso 0. Verificar entorno (activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*)

In [ ]:
import os, sys, platform
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU detectada:", gpus if gpus else "NINGUNA (el entrenamiento funcionará igual, pero más lento)")

## Paso 1. Instalar dependencias y subir el proyecto
No reinstalamos TensorFlow (Colab ya lo trae); solo lo que falta.

In [ ]:
!pip install -q ucimlrepo tf2onnx onnx onnxruntime

In [ ]:
from google.colab import files
uploaded = files.upload()   # selecciona phishing_detector.zip

In [ ]:
import zipfile, os
zip_name = [f for f in uploaded if f.endswith(".zip")][0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall(".")
os.chdir("phishing_detector")
os.makedirs("evidencias", exist_ok=True)
print("Directorio:", os.getcwd())
!ls

## Paso 2. Descargar y balancear el dataset (RNF6: ≥ 80,000 URLs, 50/50)
Fuente: **PhiUSIIL Phishing URL Dataset** (UCI, id 967; Prasad & Chandra, 2024). Convención del proyecto: `1 = phishing`, `0 = legítima`.

*Si falla la descarga* (UCI a veces responde lento): vuelve a ejecutar la celda. Si sigue fallando, copia aquí el mensaje de error y lo resolvemos.

In [ ]:
!python src/download_dataset.py --n-total 80000 --out data/raw/dataset_unificado.csv

## Paso 3. Análisis exploratorio (antes de entrenar)
Revisa balance, duplicados, longitud y **posibles atajos** (esquema http/https, prefijo `www.`, URL con ruta). Si una clase casi siempre tiene un rasgo que la otra no, el modelo puede "hacer trampa" y verse excelente sin detectar phishing real. Esto se mide en el Paso 6.

In [ ]:
!python src/eda.py --in data/raw/dataset_unificado.csv --out evidencias

In [ ]:
from IPython.display import Image, display
display(Image("evidencias/eda_longitud.png", width=620))
display(Image("evidencias/eda_atajos.png", width=680))

## Paso 4. Preprocesamiento
Tokenización por carácter (largo 200) y partición estratificada 70/10/20 con semilla 42. El test queda **reservado** hasta la evaluación final.

In [ ]:
!python src/preprocess.py --in data/raw/dataset_unificado.csv --outdir data/processed --maxlen 200

## Paso 5. Entrenamiento con validación cruzada k-fold (k = 10)
Tarda aproximadamente **20–45 minutos** con GPU. No cierres la pestaña. Si Colab te corta por cuota, usa `--k-folds 5` y lo documentamos como respaldo.

Qué esperar: al final imprime el promedio ± desviación de cada métrica y verifica RNF2 (recall > 0.85).

In [ ]:
!python src/train.py --processed-dir data/processed --models-dir models --epochs 15 --batch-size 128 --k-folds 10

## Paso 6. Evaluación en test + prueba de robustez
Matriz de confusión, ROC, precisión–recall, tabla de umbrales, errores concretos, y la **prueba de robustez** que quita esquema/`www.`/ruta del test para ver si el modelo dependía de atajos.

In [ ]:
!python src/evaluate.py --model models/cnn_phishing_final.keras --processed-dir data/processed --out evidencias

In [ ]:
from IPython.display import Image, display
display(Image("evidencias/matriz_confusion.png", width=380))
display(Image("evidencias/curva_roc.png", width=420))
import pandas as pd
print("Falsos negativos (phishing que el modelo dejó pasar) — los 10 más claros:")
display(pd.read_csv("evidencias/errores_falsos_negativos.csv").head(10))
print("Falsos positivos (legítimas marcadas como phishing) — los 10 más claros:")
display(pd.read_csv("evidencias/errores_falsos_positivos.csv").head(10))

## Paso 7. Exportar a ONNX y medir latencia (RNF1: p95 < 100 ms)
La latencia aquí es en CPU de Colab; la medición definitiva debe hacerse dentro del navegador con ONNX Runtime Web.

In [ ]:
!python src/export_onnx.py --model models/cnn_phishing_final.keras --out models/cnn_phishing.onnx --processed-dir data/processed

## Paso 8. Resumen y descarga
La celda siguiente junta lo importante en un solo bloque. **Cópialo y pégalo en el chat**: con eso redacto los resultados del informe sin inventar nada.

In [ ]:
import json
L = lambda p: json.load(open(p, encoding="utf-8"))
res = {
  "dataset": {k: L("evidencias/eda_resumen.json")[k] for k in
              ["n_total","n_phishing","n_legitimas","duplicados_exactos","casi_duplicados_sin_esquema_ni_www",
               "urls_en_hosts_repetidos_pct","pct_urls_mayores_a_200","columnas_con_alerta"]},
  "atajos": L("evidencias/eda_resumen.json")["atajos_posibles"],
  "kfold_promedio": L("models/cnn_phishing_kfold_metrics.json")["promedio"],
  "kfold_desviacion": L("models/cnn_phishing_kfold_metrics.json")["desviacion_estandar"],
  "kfold_tiempo_min": round(L("models/cnn_phishing_kfold_metrics.json")["tiempo_total_segundos"]/60,1),
  "test": L("evidencias/metricas_test.json"),
  "robustez": L("evidencias/robustez.json"),
  "latencia_onnx": L("models/latencia_onnx_metrics.json"),
  "entorno": {"tensorflow": tf.__version__, "gpu": bool(gpus)},
}
print(json.dumps(res, ensure_ascii=False, indent=1))

In [ ]:
import shutil
from google.colab import files
shutil.copy("data/processed/vocab_meta.json", "models/vocab_meta.json")
os.makedirs("pack", exist_ok=True)
shutil.copytree("models", "pack/models", dirs_exist_ok=True)
shutil.copytree("evidencias", "pack/evidencias", dirs_exist_ok=True)
shutil.make_archive("fase_A_resultados", "zip", "pack")
files.download("fase_A_resultados.zip")

### Si algo falla
- **Error al instalar/importar `tf2onnx` o al exportar ONNX:** copia el mensaje completo; el script ya tiene una ruta alternativa automática, pero depende de la versión de TensorFlow de Colab.
- **Colab se desconecta durante el Paso 5:** vuelve a conectar, repite desde el Paso 1 (los archivos se pierden) y usa `--k-folds 5`.
- **Descarga de UCI falla varias veces:** descarga el CSV manualmente desde https://archive.ics.uci.edu/dataset/967 y avísame.